In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from sklearn.base import clone

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load data and firt look

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
sample_submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
num_columns = [i for i in train.columns if train[i].dtype in ['int64','float64']]
cat_columns = train.select_dtypes(include=['object']).columns.tolist()
print(f'numerical columns   : {num_columns} \ncategorical columns : {cat_columns}')

In [ ]:
for i in cat_columns:
    if i == 'Name':
        continue
    print(f'column name : {i}\n {i}s unique values  {train[i].unique()}\n')

# Missing values

In [ ]:
missing_values = train.isnull().sum()
missing_percent = (missing_values/len(train))*100
missing_df = pd.DataFrame({'missing values':missing_values, 'missing_percent':missing_percent, 'missing dtype':train.dtypes})
missing_df

I tried to fill in the age column by analyzing its correlations and patterns,
but since i will be using XGBoost idecided to fill the Cabin column with Unknown instead

In [ ]:
df = train.copy()

df["Sex_num"] = df["Sex"].map({"male": 1, "female": 0})
df["Embarked"] = df["Embarked"].fillna("Unknown")
df["Cabin_Missing"] = df["Cabin"].isna().astype(int)

sub = df.dropna(subset=["Age"])

# corr
cols = ["Survived", "Pclass", "SibSp", "Parch", "Fare", "Sex_num", "Cabin_Missing"]
corr = sub[["Age"] + cols].corr().loc["Age"].drop("Age").sort_values(key=abs, ascending=False)

print("=== Age ile korelasyonlar ===")
print(corr)

# graph 1 age istribution
plt.hist(sub["Age"], bins=30)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

# graph 2 age vs fare
plt.scatter(sub["Fare"], sub["Age"], alpha=0.5)
plt.title("Age vs Fare")
plt.xlabel("Fare")
plt.ylabel("Age")
plt.show()

# graph 3 sex vs age
male_age = sub.loc[sub['Sex'] == 'male','Age']
female_age = sub.loc[sub['Sex'] == 'female','Age']

plt.hist(male_age, bins=20, alpha=0.6, label='Male')
plt.hist(female_age, bins=20, alpha=0.6, label='Female')

plt.title('Age distrubution by sex')
plt.xlabel('Age')
plt.ylabel('count')
plt.legend()
plt.show()

#graph 4 parch vs age
avg_age = train.groupby("Parch")["Age"].mean()

plt.bar(avg_age.index, avg_age.values)
plt.title("Average Age by Parch")
plt.xlabel("Parch (Number of Parents/Children Aboard)")
plt.ylabel("Average Age")
plt.show()


Age Distribution:
Most passengers were between 20–30 years old, showing a young passenger population.

Age vs Fare:
There’s no clear relationship between age and fare

Age Distribution by Sex:
Male passengers are more numerous and slightly younger on average than females

Average Age by Parch:
Passengers with more parents/children aboard tend to be older on average



Based on the analysis of Age distributions and its relationship with features like Parch, Sex, and Pclass, I filled the missing Age values using grouped medians to preserve logical patterns in the data and avoid random imputation

In [ ]:
import pandas as pd
import numpy as np

def fill_missing_age(df):
    df["Age_was_missing"] = df["Age"].isna().astype(int)
    
    med_table = (
        df.groupby(["Parch", "Sex", "Pclass"])["Age"]
          .median()
          .reset_index()
          .rename(columns={"Age": "Age_median"})
    )

    df = df.merge(med_table, on=["Parch", "Sex", "Pclass"], how="left")
    df["Age"] = df["Age"].fillna(df["Age_median"])
    
    sex_median = df.groupby("Sex")["Age"].transform("median")
    
    df["Age"] = df["Age"].fillna(sex_median)
    
    df["Age"] = df["Age"].fillna(df["Age"].median())
    
    df.drop(columns=["Age_median"], inplace=True)
    
    return df

train = fill_missing_age(train)
test = fill_missing_age(test)

# Basic data analysis

In [ ]:
plt.figure(figsize=(8,4))
df["Survived"].value_counts().plot(kind="bar")
plt.title("Overall Survival Count")
plt.xlabel("Survived (0=No, 1=Yes)")
plt.ylabel("Count")
plt.show()


survival_by_sex = df.groupby("Sex")["Survived"].mean()
plt.bar(survival_by_sex.index, survival_by_sex.values, color=["pink", "steelblue"])
plt.title("Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Survival Rate")
plt.show()


sex_counts = df["Sex"].value_counts()

plt.bar(sex_counts.index, sex_counts.values, color=["steelblue", "pink"])
plt.title("Passenger Count by Sex")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.show()


plt.figure(figsize=(6,4))
plt.hist(df[df["Survived"] == 1]["Age"].dropna(), bins=30, color="lightgreen", alpha=0.7, edgecolor="black")
plt.title("Age Distribution of Survivors")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

survival_by_class = df.groupby("Pclass")["Survived"].mean()
plt.bar(survival_by_class.index.astype(str), survival_by_class.values)
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Pclass (1=High, 3=Low)")
plt.ylabel("Survival Rate")
plt.show()


df["Embarked"] = df["Embarked"].fillna("Unknown")
survival_by_emb = df.groupby("Embarked")["Survived"].mean()
plt.bar(survival_by_emb.index, survival_by_emb.values)
plt.title("Survival Rate by Embarkation Port")
plt.xlabel("Embarked (C=Cherbourg, Q=Queenstown, S=Southampton)")
plt.ylabel("Survival Rate")
plt.show()


plt.hist(df[df['Survived']== 1]['Fare'], bins=10 ,color="lightgreen", alpha=0.7, edgecolor="black")
plt.title("Survival vs Fare")
plt.xlabel("Fare")
plt.ylabel("Survived (0/1)")
plt.show()

ChatGPT explains why women survived more than men like this


“Women and children first” policy: Maritime customs and Titanic’s evacuation orders prioritized women/children. Officers and crew actively enforced this on many lifeboats.

Lifeboat capacity & access: Lifeboats were insufficient and located mainly on upper decks; passengers housed near public spaces (often 1st class) reached them earlier. Many women were in these groups.

Social norms & group behavior: Women traveling with families were more readily escorted into boats; some men yielded seats due to prevailing norms and stayed aboard.

Timing & location effects: The collision occurred at night; crowding, class partitions (doors/stairways), and deck proximity shaped access. Those already near the boat deck gained an early advantage.

Statistical confounders: The gender gap intertwines with Pclass, age, cabin location, and embarkation port. A higher share of 1st-class among females can amplify the observed gap.

# Encoding  - Cabin , Ticket and Name 

In [ ]:
import re
def process_ticket_cabin(df):
    # === TICKET ===
    ticket = df['Ticket'].fillna('Unknown').astype(str)

    # Ticket prefix 
    df['Ticket_Prefix'] = ticket.str.extract(r'([A-Za-z\.\/]+)', expand=False).str.replace(r'[\.\/]', '', regex=True).str.upper()
    df['Ticket_Prefix'] = df['Ticket_Prefix'].fillna('NONE')

    # Ticket number 
    df['Ticket_Number'] = ticket.str.extract(r'(\d+)', expand=False).astype(float)

    # Ticket length 
    df['Ticket_Len'] = ticket.str.len()

    # Ticket group size 
    df['Ticket_GroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')

    # Ticket prefix frequency encoding
    prefix_freq = df['Ticket_Prefix'].value_counts(normalize=True)
    df['Ticket_Prefix_Freq'] = df['Ticket_Prefix'].map(prefix_freq)

    # === CABIN ===
    cabin = df['Cabin'].fillna('Unknown').astype(str)

    # first cabin
    first_cabin = cabin.str.split().str[0]

    # Deck 
    df['Deck'] = first_cabin.str[0].replace('U', 'Unknown')

    # Deck ordinal 
    deck_map = {**{chr(ord('A')+i): i+1 for i in range(7)}, 'T': 8, 'U': 0}
    df['Deck_Ordinal'] = df['Deck'].map(deck_map).fillna(0).astype(int)

    # cabin num
    df['Cabin_Number'] = pd.to_numeric(first_cabin.str[1:].str.extract(r'(\d+)')[0], errors='coerce')

    # count cabin
    df['Cabin_Count'] = cabin.str.split().str.len()

    # missing cabin
    df['Cabin_Missing'] = (df['Cabin'] == 'Unknown').astype(int)

    # Deck frekans encoding
    deck_freq = df['Deck'].value_counts(normalize=True)
    df['Deck_Freq'] = df['Deck'].map(deck_freq)

    # === DROP RAW COLUMNS ===
    df = df.drop(columns=['Ticket', 'Cabin'], errors='ignore')

    return df

In [ ]:
train = process_ticket_cabin(train)
test  = process_ticket_cabin(test)

In [ ]:
# we get Name title
def process_name_title(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    title = df['Name'].astype(str).str.extract(r',\s*([^\.]+)\.', expand=False)
    title = title.str.strip().str.title()

    title = title.replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = {'Mr', 'Mrs', 'Miss', 'Master'}
    title = title.where(title.isin(common_titles), other='Rare')

    df['Title'] = title.fillna('Unknown').astype('category')
    return df

In [ ]:
train = process_name_title(train)
test = process_name_title(test)

In [ ]:
train["Title"] = train["Title"].astype("object")
test["Title"] = test["Title"].astype("object")

In [ ]:
train["Cabin_Number"] = train["Cabin_Number"].fillna(train['Cabin_Number'].mean())
test["Cabin_Number"]  = test["Cabin_Number"].fillna(test['Cabin_Number'].mean())

train["Fare"] = train["Fare"].fillna(train["Fare"].mean())
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

test["Fare"] = test["Fare"].fillna(test["Fare"].mean())
test["Embarked"] = test["Embarked"].fillna(test["Embarked"].mode()[0])

train['Ticket_Number'] = train['Ticket_Number'].fillna(train['Ticket_Number'].mean())
test['Ticket_Number'] = test['Ticket_Number'].fillna(test['Ticket_Number'].mean())

In [ ]:
train.drop('PassengerId', axis=1, inplace=True)
test.drop('PassengerId',axis=1,inplace=True)

train.drop('Name', axis=1 , inplace=True)
test.drop('Name', axis=1, inplace=True)

# Model 

In [ ]:
target = 'Survived'

In [ ]:
cat_columns = train.select_dtypes(include=['object']).columns.tolist()
num_columns = [i for i in train.columns if train[i].dtype in ['int64','float64'] and i != target]

In [ ]:
print(f'categorical columns    : {cat_columns}\nnumerical columns      : {num_columns}')

In [ ]:
X = train.drop(target,axis=1)
y = train[target]

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
preprocess = ColumnTransformer(
    transformers=[
        ("ohe", ohe, cat_columns),
        ("num", "passthrough", num_columns)
    ],
    remainder="drop"
)


# def Onhe(df,cat_columns):
#     ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#     encoded = ohe.fit_transform(df[cat_columns])
#     encoded_cols = ohe.get_feature_names_out(cat_columns)
#     encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

#     train_ohe = pd.concat([train.drop(columns=cat_columns), encoded_df], axis=1)

In [ ]:
pos = y.sum()
neg = len(y) - pos
spw = neg / max(pos, 1)

xgb_params = dict(
    n_estimators=20000,        
    learning_rate=0.009,
    max_depth=3,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    eval_metric="auc",
    tree_method="hist",
    random_state=24,
    n_jobs=-1,
    scale_pos_weight=spw
)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=24)


oof = np.zeros(len(X), dtype=float)
fold_scores = []
models = []

In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    prep = clone(preprocess)
    prep.fit(X_train)

    X_tr_t  = prep.transform(X_train)
    X_val_t = prep.transform(X_val)
    
    model = XGBClassifier(**xgb_params, early_stopping_rounds=500)
    model.fit(
        X_tr_t, y_train,
        eval_set=[(X_val_t, y_val)],
        verbose=False
    )

    val_pred = model.predict_proba(X_val_t)[:, 1]
    oof[val_idx] = val_pred
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_scores.append(fold_auc)
    print(f"[Fold {fold}] AUC: {fold_auc:.5f} | best_iteration={model.best_iteration}")

    models.append(model)

print(f"OOF AUC: {roc_auc_score(y, oof):.5f}")
print("Fold AUCs:", [f"{s:.5f}" for s in fold_scores])

# Submission

In [ ]:
best_t = max(np.linspace(0.3, 0.7, 81), key=lambda t: ((oof >= t).astype(int) == y.values).mean())
print("Best threshold:", best_t)

test_proba = np.zeros(len(test), dtype=float)
# fold 0 and fold 4 low auc score so
for fold, (tr, _) in enumerate(skf.split(X, y)):
    if fold == 4:  
        break
    if fold ==0:
        continue
    prep = clone(preprocess).fit(X.iloc[tr])
    
    X_te_t = prep.transform(test[cat_columns + num_columns])
    test_proba += models[fold].predict_proba(X_te_t)[:, 1]

test_proba /= 4

submission = pd.DataFrame({
    "PassengerId": pd.read_csv("/kaggle/input/titanic/test.csv")["PassengerId"],
    "Survived": (test_proba >= best_t).astype(int)
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
ls